# Embeddings & Vector Store
---

# Información

## Objetivo

Generar embeddings y construir un vector store utilizando información oficial del Plan Comunal de Emergencia de Valdivia.

## Objetivos específicos

- Cargar el documento limpio generado en la etapa de ingestión.
- Crear fragmentos de texto para búsqueda semántica.
- Transformar los fragmentos en embeddings.
- Construir un índice vectorial.

## Entradas

- data/processed/Plan-Comunal-de-Emergencia-2025-2.json

## Salidas

- Embeddings generados.
- Vector store para recuperación semántica.



## Etapa 1: Generación de Embeddings

En esta etapa se transformará el contenido textual de cada `Chunk` en una representación vectorial utilizando un modelo de Sentence Transformers.

Los embeddings permiten representar el significado semántico del texto mediante vectores numéricos. Estas representaciones constituyen la base de la búsqueda semántica utilizada por el sistema RAG.

### 1. Importación de dependencias.

In [8]:
from pathlib import Path
import sys

# Agregar la raíz del proyecto al PATH para importar módulos de src
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [10]:
from sentence_transformers import SentenceTransformer

from src.ingestion.loaders import load_pdf
from src.processing.cleaner import clean_document
from src.processing.embedder import generate_embeddings
from src.processing.chunker import create_chunks


### 2. Carga del documento

En esta sección se carga un documento PDF de prueba utilizando el módulo `load_pdf`, desarrollado durante la etapa de Ingesta. El objetivo es obtener un objeto `Document` que servirá como punto de partida para el resto del pipeline.

Para esta validación se utilizará el documento **test_plan_comunal.pdf**, el mismo empleado en las pruebas de las etapas anteriores, garantizando la consistencia durante el desarrollo del proyecto.

In [11]:
import json
from pathlib import Path


processed_file = Path(
    "../data/processed/Plan-Comunal-de-Emergencia-2025-2.json"
)

with processed_file.open(
    "r",
    encoding="utf-8"
) as file:
    document_data = json.load(file)


document_data.keys()

dict_keys(['id', 'title', 'source', 'file_type', 'content', 'metadata'])

In [12]:
from src.models.document import Document


document = Document(
    id=document_data["id"],
    title=document_data["title"],
    source=document_data["source"],
    file_type=document_data["file_type"],
    content=document_data["content"],
    metadata=document_data["metadata"]
)

print(f"Título: {document.title}")
print(f"Fuente: {document.source}")
print(f"Tipo de archivo: {document.file_type}")
print(f"Longitud del contenido: {len(document.content)} caracteres")
print(f"Metadatos: {document.metadata}")



Título: Plan-Comunal-de-Emergencia-2025-2
Fuente: ..\data\raw\muni_valdivia\planes\Plan-Comunal-de-Emergencia-2025-2.pdf
Tipo de archivo: pdf
Longitud del contenido: 90189 caracteres
Metadatos: {'pages': 33}


---
## Etapa 2. Preparación de chunks

In [13]:
import src.processing.chunker as chunker

print(chunker.__file__)
print(hasattr(chunker, "create_chunks"))
print(dir(chunker))

c:\ALURA_AI_Proyect\Challege_Alura_ONE_AI_FOR_TECH\Challenge_Alura_Agente\src\processing\chunker.py
True
['Chunk', 'Document', 'RecursiveCharacterTextSplitter', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'create_chunks', 'uuid4']


In [14]:
print(f"Cantidad de chunks generados: {len(chunks)}")

primer_chunk = chunks[0]

print(f"ID: {primer_chunk.id}")
print(f"Documento: {primer_chunk.document_id}")
print(f"Índice: {primer_chunk.chunk_index}")
print(f"Longitud: {len(primer_chunk.content)} caracteres")

print("\nPrimeros 300 caracteres:\n")
print(primer_chunk.content[:300])

NameError: name 'chunks' is not defined

In [17]:
chunks = chunker.create_chunks(document)
len(chunks)

229

In [18]:
print(f"Cantidad de chunks: {len(chunks)}")

primer_chunk = chunks[0]

print(f"ID: {primer_chunk.id}")
print(f"Documento origen: {primer_chunk.document_id}")
print(f"Índice: {primer_chunk.chunk_index}")
print(f"Longitud: {len(primer_chunk.content)} caracteres")

print("\nPrimeros 300 caracteres:\n")
print(primer_chunk.content[:300])

Cantidad de chunks: 229
ID: 53dbfa2e-b975-4cbd-8256-49944b0ca221
Documento origen: 12e5a59e-eaff-45e6-a141-ab890d253c69
Índice: 0
Longitud: 499 caracteres

Primeros 300 caracteres:

Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página 
1 de 33 
Fecha: Agosto-2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
PLAN COMUNAL DE EMERGENCIA 
VALDIVIA 
 
 
 
ANEXO – PLAN POR AMENAZA INCENDIO FORESTAL 
ANEXO - PLAN POR AMENAZA 


---
## Etapa 3. Generación de embeddings

### 3.1  Carga de modelo de embeddings

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2758.90it/s]


### 3.2  Importar el módulo

In [20]:
import src.processing.embedder as embedder

### 3.3 Generación de embeddings para cada chunk 

In [21]:
embedded_chunks = embedder.generate_embeddings(
    chunks=chunks,
    model=model,
)

### 3.4  Validar el resultado

In [22]:
print(f"Chunks con embeddings: {len(embedded_chunks)}")

Chunks con embeddings: 229


### 3.5  Validar el primer embedding generado

In [23]:
primer_chunk = embedded_chunks[0]

print(type(primer_chunk.embedding))
print(len(primer_chunk.embedding))

<class 'numpy.ndarray'>
384


### 3.6  Validación adicional

In [24]:
todos_tienen_embedding = all(
    chunk.embedding is not None
    for chunk in embedded_chunks
)

print(f"Todos los chunks tienen embedding: {todos_tienen_embedding}")

Todos los chunks tienen embedding: True


In [25]:
primer_embedding = embedded_chunks[0].embedding

print(type(primer_embedding))
print(primer_embedding.shape)
print(primer_embedding[:10])

<class 'numpy.ndarray'>
(384,)
[ 0.08619918  0.00378665  0.01216879  0.02605192 -0.01683555 -0.05457666
 -0.1163444   0.0337018  -0.03534188  0.03167299]


---
## Etapa 4. Construcción del vector store (CHROMADB)

ETAPA 1 - Ingestion
        ✅ Document

ETAPA 2 - Chunking
        ✅ 229 Chunk

ETAPA 3 - Embeddings
        ✅ 229 vectores semánticos

ETAPA 4 - Vector Store (CHROMADB)
        🚧 En desarrollo

Objetivo de esta etapa

Construir un índice vectorial que permita almacenar los embeddings generados y realizar búsquedas por similitud semántica.

### 1. Persistencia del vector store

In [33]:
import chromadb

import src.processing.vector_store as vector_store

### 2. Crear la colección ChromaDB

In [39]:
collection = vector_store.get_or_create_collection(
    collection_name="sophia_emergencias_valdivia"
)

### 3. Verificar la colección

In [40]:
print(collection.name)

sophia_emergencias_valdivia


### 4. Indexar los chunks con embeddings

In [41]:
vector_store.index_chunks(
    chunks=embedded_chunks,
    collection=collection,
)

### 5. Validar la cantidad almacenada

In [42]:
collection.count()

229